## `make_hscy3_fromcats.ipynb`

--------------------------------

This notebook extracts data from `/pscratch/sd/z/ztq1996/HSCDATA/HSCY3` and asserts it is the same as the data provided by the generated catalog with `make_hscy3.py`.

In [10]:
# imports
import fitsio as fio 
import numpy as np 
import healpy as hp
import cmocean.cm as cmo
import matplotlib.pyplot as plt
import tqdm

from numpy.lib import recfunctions as rfn
from astropy.table import Table, vstack
from pathlib import Path

CAT_DIR = Path('/global/cfs/projectdirs/desi/users/jchdj/desi-y3-hsc/data/hsc/cat/')
hsc_cat = CAT_DIR / 'hscy3_cat.fits'
hsc_patch_list = sorted(list(Path('/pscratch/sd/z/ztq1996/HSCDATA/HSCY3').glob('*fits')))

In [11]:
tbl=[]
for patch in tqdm.tqdm(hsc_patch_list, 'Concatenating HSC patches'): 
    tbl.append(
        Table(
            fio.FITS(patch)[1].read(
            columns=[
                'object_id', 
                'i_ra', 
                'i_dec', 
                'i_cmodel_mag', 
                'i_cmodel_magerr', 
                'i_cmodel_flag', 
                'dnnz_photoz_best'
                ]
            )
        )
    )
tblfull = vstack(tbl)

tblfull.write(CAT_DIR / 'hsc_y3_from_patches.fits', format='fits', overwrite=True)

Concatenating HSC patches: 100%|██████████| 6/6 [00:10<00:00,  1.75s/it]


In [ ]:
ra_uncut = tblfull['i_ra']
dec_uncut = tblfull['i_dec']

all_ra_length = len(ra_uncut)
all_dec_length = len(dec_uncut)
# returns True if the pixel is not in the area we want to remove
gama09H_mask =  ~(
    (1.6 < dec_uncut) 
    & (dec_uncut < 5) 
    & (132.5 < ra_uncut) 
    & (ra_uncut < 140)
    )

ra_cut = ra_uncut[gama09H_mask]
dec_cut = dec_uncut[gama09H_mask]

print(
    f"Cutting the catalog from {all_ra_length} to {len(ra_cut)} "
    f"[reduction of {1 - len(ra_cut) / all_ra_length:.2%}]"
    )

Cutting the catalog from 35805482 to 34407034 [reduction of 3.91%]
